# Zero-shot text-to-tracklet benchmark

Frozen image and video language models, scored zero-shot on three caption-to-person retrieval datasets. Every caption is a query; its positives are the gallery items of the same identity.

| Dataset | Gallery | Queries | What the captions describe |
|---|---|---|---|
| **TVPReid** (`prid`, `ilids`, `duke`) | 820 test videos | 2 per video | appearance, some motion and scene |
| **RSTPReid** (`test`) | 1,000 still images, 200 IDs | 2 per image | appearance (MSMT17 cameras) |
| **GroOT-MOT17** (`all`, `appearance`, `action`, `combined`) | one mp4 per MOT17 track | 1 appearance + 1 action per track, plus 1 refined combined sentence | `appearance`, `action`, and `combined` are scored separately |

CUHK-PEDES is excluded because IRRA was trained on it; every dataset here is zero-shot for every model.

**How to run**

1. GPU T4, Internet on. Add a Kaggle secret named `hugging_face` (needed for the gated InternVideo2-1B-s2 checkpoint in the `base` group).
2. Pick `ENV_GROUP` in the configuration cell. The groups need different `transformers` versions, so each group is its own Kaggle session:
   - `base`: SigLIP 2, Perception Encoder, IRRA, InternVideo2-1B-s2, X-CLIP, LanguageBind, InternVideo2 CLIP-S
   - `extended_modern`: OpenAI CLIP ViT-L/14, Jina CLIP v2, Qwen3-VL-Embedding-2B
   - `extended_gme`: GME-Qwen2-VL-2B
3. Run All. Every result row is written as JSON the moment it finishes, so a restarted session skips what is done.
4. To combine groups, save each session's `/kaggle/working/zero_shot_results` as a Kaggle dataset and attach it; the tables section merges every results folder it finds.

## 1. Configuration

In [ ]:
from pathlib import Path

ENV_GROUP = "base"  # "base", "extended_modern", or "extended_gme" (one Kaggle session each)
RUN_DATASETS = ["tvpreid", "rstpreid", "groot_mot17"]

MODEL_GROUPS = {
    "base": [
        "siglip2",
        "pe_core_l14",
        "irra",
        "internvideo2_s2_1b",
        "xclip",
        "languagebind",
        "internvideo2",
    ],
    "extended_modern": ["openai_clip_vit_l14", "jina_clip_v2", "qwen3_vl_embed_2b"],
    "extended_gme": ["gme_qwen2_vl_2b"],
}
MODELS = MODEL_GROUPS[ENV_GROUP]

# "video": single-clip and sliding-window protocols. "image": one still per item,
# so only the single-clip protocol (the frame is repeated to fill the clip).
DATASETS = {
    "tvpreid": {"label": "TVPReid", "subsets": ["prid", "ilids", "duke"], "protocols": "video"},
    "rstpreid": {"label": "RSTPReid", "subsets": ["test"], "protocols": "image"},
    "groot_mot17": {"label": "GroOT-MOT17", "subsets": ["all", "appearance", "action", "combined"], "protocols": "video"},
}
SPLIT = "test"
DEVICE = "cuda"
BATCH = 4
TEXT_BATCH = 16
POOLS = ("mean", "mean_s8", "max", "query_max")

# 1B-s2 is built for 4-frame clips; every other model uses 8.
NATIVE_FRAMES = {"internvideo2_s2_1b": 4}
MODEL_BATCH = {
    "siglip2": 1,
    "pe_core_l14": 1,
    "irra": 2,
    "internvideo2_s2_1b": 1,
    "internvideo2": 2,
    "languagebind": 2,
    "openai_clip_vit_l14": 4,
    "jina_clip_v2": 1,
    "gme_qwen2_vl_2b": 1,
    "qwen3_vl_embed_2b": 1,
}
MODEL_TEXT_BATCH = {"jina_clip_v2": 8, "gme_qwen2_vl_2b": 8, "qwen3_vl_embed_2b": 1}

SINGLE_PROTOCOLS = [
    {"name": "uniform8", "num_frames": 8, "frame_sample": "uniform", "family": "clip8"},
    {"name": "uniform4", "num_frames": 4, "frame_sample": "uniform", "family": "s2"},
    {"name": "middle4", "num_frames": 4, "frame_sample": "middle", "family": "s2", "video_only": True},
]
WINDOW_PROTOCOLS = [
    {"name": "vt_1fps_n12", "sample_fps": 1.0, "max_frames": 12, "stride": 4},
    {"name": "vt_2fps_n32", "sample_fps": 2.0, "max_frames": 32, "stride": 4},
    {"name": "reid_8fps_n64", "sample_fps": 8.0, "max_frames": 64, "stride": 4},
]

PRETTY = {
    "irra": "IRRA",
    "pe_core_l14": "Perception Encoder L/14",
    "siglip2": "SigLIP 2 So400m",
    "internvideo2": "InternVideo2 CLIP-S",
    "internvideo2_s2_1b": "InternVideo2-1B-s2",
    "languagebind": "LanguageBind",
    "xclip": "X-CLIP",
    "openai_clip_vit_l14": "OpenAI CLIP L/14",
    "jina_clip_v2": "Jina CLIP v2",
    "gme_qwen2_vl_2b": "GME-Qwen2-VL-2B",
    "qwen3_vl_embed_2b": "Qwen3-VL-Embedding-2B",
}

WORK = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path("results")
RESULTS_ROOT = WORK / "zero_shot_results"
RESULTS_DIR = RESULTS_ROOT / ENV_GROUP
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
unknown = [d for d in RUN_DATASETS if d not in DATASETS]
if unknown:
    raise ValueError(f"Unknown datasets {unknown}; choose from {list(DATASETS)}")
print("Group", ENV_GROUP, "models", MODELS, flush=True)
print("Datasets", RUN_DATASETS, "results", RESULTS_DIR, flush=True)

## 2. Setup

Clones the repo (and OpenGVLab/InternVideo when 1B-s2 is in the group), installs the group's dependencies, checks the package version, and loads the Hugging Face token. If `transformers` was already imported with a different version, restart the session and run again.

In [ ]:
import os
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version

SHAWAF_URL = "https://github.com/BASSAT-BASSAT/Benchmarking-Video-Image-language-models-for-Tracklet-retrieval-.git"
INTERNVIDEO_URL = "https://github.com/OpenGVLab/InternVideo.git"
EXPECTED = "0.1.18"
TORCH_INDEX = "https://download.pytorch.org/whl/cu126"


def run(cmd, cwd=None):
    print("+", " ".join(map(str, cmd)), flush=True)
    subprocess.check_call([str(part) for part in cmd], cwd=str(cwd) if cwd else None)


def installed(name):
    try:
        return version(name)
    except PackageNotFoundError:
        return None


def sync_repo(url, path):
    if path.exists():
        run(["git", "fetch", "origin"], cwd=path)
        run(["git", "reset", "--hard", "origin/main"], cwd=path)
    else:
        run(["git", "clone", url, str(path)])


NEEDS_INTERNVIDEO = "internvideo2_s2_1b" in MODELS
ON_KAGGLE = Path("/kaggle/working").is_dir()
transformers_before = installed("transformers")
transformers_loaded = "transformers" in sys.modules

if ON_KAGGLE:
    shawaf_dir = Path("/kaggle/working/shawaf-vlm")
    intern_dir = Path("/kaggle/working/InternVideo")
    sync_repo(SHAWAF_URL, shawaf_dir)
    if NEEDS_INTERNVIDEO:
        sync_repo(INTERNVIDEO_URL, intern_dir)
    if ENV_GROUP == "base":
        run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{shawaf_dir}[all]"])
        run(
            [
                sys.executable, "-m", "pip", "install", "-q",
                "einops", "timm", "av", "imageio", "librosa", "soundfile",
                "pandas", "pyyaml", "scipy", "wandb", "gdown",
            ]
        )
        try:
            run([sys.executable, "-m", "pip", "install", "-q", "decord"])
        except subprocess.CalledProcessError:
            print("decord wheel failed; the 1B-s2 path stubs it and uses PyAV", flush=True)
    else:
        if not (installed("torch") or "").startswith("2.8.0"):
            run([sys.executable, "-m", "pip", "install", "-q", "--index-url", TORCH_INDEX, "torch==2.8.0", "torchvision==0.23.0"])
        run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{shawaf_dir}[{ENV_GROUP}]", "gdown>=5.2", "pandas"])
else:
    here = Path.cwd().resolve()
    shawaf_dir = here
    for candidate in [here, *here.parents]:
        if (candidate / "shawaf_vlm").is_dir() and (candidate / "pyproject.toml").is_file():
            shawaf_dir = candidate
            break
    intern_dir = shawaf_dir / "InternVideo"
    if NEEDS_INTERNVIDEO and not intern_dir.is_dir():
        run(["git", "clone", INTERNVIDEO_URL, str(intern_dir)], cwd=shawaf_dir)

transformers_after = installed("transformers")
if transformers_loaded and transformers_after != transformers_before:
    raise RuntimeError(
        f"transformers changed from {transformers_before} to {transformers_after} after it was "
        "imported. Restart the session, then Run All again."
    )

if NEEDS_INTERNVIDEO:
    os.environ["INTERNVIDEO_ROOT"] = str(intern_dir)
if str(shawaf_dir) not in sys.path:
    sys.path.insert(0, str(shawaf_dir))
for name in list(sys.modules):
    if name == "shawaf_vlm" or name.startswith("shawaf_vlm."):
        del sys.modules[name]

import shawaf_vlm

print("shawaf_vlm", shawaf_vlm.__version__, shawaf_vlm.__file__, flush=True)
print("transformers", transformers_after, flush=True)
if shawaf_vlm.__version__ != EXPECTED:
    raise RuntimeError(
        f"Expected shawaf_vlm {EXPECTED}, found {shawaf_vlm.__version__}. "
        "Restart the session and run this cell again after origin/main updates."
    )

token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
if not token and ON_KAGGLE:
    try:
        from kaggle_secrets import UserSecretsClient

        token = UserSecretsClient().get_secret("hugging_face")
    except Exception as exc:
        print("No hugging_face secret:", type(exc).__name__, flush=True)
if token:
    os.environ["HF_TOKEN"] = token
    os.environ["HUGGING_FACE_HUB_TOKEN"] = token
    from huggingface_hub import login

    login(token=token, add_to_git_credential=False)
    print("Hugging Face token loaded", flush=True)
elif NEEDS_INTERNVIDEO:
    raise RuntimeError(
        "Hugging Face token missing. InternVideo2-1B-s2 is gated: add a Kaggle secret "
        "named hugging_face and attach it to this notebook."
    )
else:
    print("No Hugging Face token; public datasets and models only", flush=True)

## 3. Datasets

TVPReid and GroOT-MOT17 download only their test mp4s from the Hugging Face Hub (`bassatbassat/TVPReid`, `bassatbassat/GroOT-MOT17`). RSTPReid downloads its archive from the authors' Google Drive, or uses an attached Kaggle input that contains `data_captions.json` and `imgs/`.

In [ ]:
import pandas as pd
from IPython.display import Markdown, display

from shawaf_vlm.data import (
    discover_local_groot,
    download_groot,
    download_rstpreid,
    download_tvpreid,
    load_groot,
    load_rstpreid,
    load_tvpreid,
)

DATA_ROOTS = {}
if "tvpreid" in RUN_DATASETS:
    DATA_ROOTS["tvpreid"] = download_tvpreid(configs=tuple(DATASETS["tvpreid"]["subsets"]), split=SPLIT)
if "rstpreid" in RUN_DATASETS:
    DATA_ROOTS["rstpreid"] = download_rstpreid()
if "groot_mot17" in RUN_DATASETS:
    DATA_ROOTS["groot_mot17"] = discover_local_groot() or download_groot(
        configs=tuple(DATASETS["groot_mot17"]["subsets"]), split=SPLIT
    )


def load_split(dataset, subset):
    root = DATA_ROOTS[dataset]
    if dataset == "tvpreid":
        return load_tvpreid(subset, split=SPLIT, root=root)
    if dataset == "rstpreid":
        return load_rstpreid(root, split=subset)
    if dataset == "groot_mot17":
        return load_groot(subset, split=SPLIT, root=root)
    raise ValueError(dataset)


summary = []
for dataset in RUN_DATASETS:
    for subset in DATASETS[dataset]["subsets"]:
        splits = load_split(dataset, subset)
        per_id = pd.Series([g.person_id for g in splits.gallery]).value_counts()
        positives = pd.Series([per_id.get(q.person_id, 0) for q in splits.query])
        summary.append(
            {
                "dataset": DATASETS[dataset]["label"],
                "subset": subset,
                "queries": len(splits.query),
                "gallery": len(splits.gallery),
                "identities": int(per_id.size),
                "positives / query": round(float(positives.mean()), 2),
                "root": str(DATA_ROOTS[dataset]),
            }
        )
DATASET_SUMMARY = pd.DataFrame(summary)
display(DATASET_SUMMARY)

## 4. Run

Order: model, then dataset, then protocol, then subset. Each finished row is saved to `zero_shot_results/<group>/<dataset>/<model>_<subset>_<protocol>_<pool>.json` and skipped on rerun.

GroOT's four subsets share one gallery, so gallery embeddings are reused across `all`, `appearance`, `action`, and `combined` within a protocol (those rows are marked `gallery_cached`; speed is read from the uncached row).

In [ ]:
import gc
import json
import traceback
from datetime import datetime, timezone

from shawaf_vlm.eval_loop import evaluate_text_to_tracklet, evaluate_text_to_tracklet_windows
from shawaf_vlm.metrics import format_metrics
from shawaf_vlm.models import build_encoder
from shawaf_vlm.models.runtime import ensure_cuda_healthy

METRIC_KEYS = [
    "Rank-1", "Rank-5", "Rank-10", "Rank-20", "Rank-50",
    "mAP", "MdR", "MnR", "nDCG@10", "mINP",
    "num_valid_queries", "num_gallery", "num_clips",
    "decode_s", "video_s", "text_s", "score_s", "total_s",
    "video_ms_per_item", "peak_gpu_gb", "reserved_gpu_gb",
]


class StopModel(Exception):
    """CUDA OOM: stop this model, keep benchmark settings unchanged."""


class GalleryCache:
    """Reuse the last gallery embedding when the exact same frame lists come back."""

    def __init__(self, encoder):
        self.encoder = encoder
        self._key = None
        self._features = None
        self.last_hit = False

    def encode_videos(self, videos, batch_size=4):
        key = tuple(tuple(str(path) for path in clip) for clip in videos)
        if key == self._key:
            self.last_hit = True
            return self._features
        self.last_hit = False
        self._features = self.encoder.encode_videos(videos, batch_size=batch_size)
        self._key = key
        return self._features

    def encode_texts(self, texts, batch_size=32):
        return self.encoder.encode_texts(texts, batch_size=batch_size)

    def __getattr__(self, name):
        return getattr(self.encoder, name)


def family(model):
    return "s2" if model in NATIVE_FRAMES else "clip8"


def protocols_for(model, dataset):
    kind = DATASETS[dataset]["protocols"]
    specs = []
    for spec in SINGLE_PROTOCOLS:
        if spec["family"] != family(model):
            continue
        if kind == "image" and spec.get("video_only"):
            continue
        specs.append({**spec, "type": "single"})
    if kind == "video":
        specs.extend({**spec, "type": "window"} for spec in WINDOW_PROTOCOLS)
    return specs


def sampling_text(model, spec):
    if spec["type"] == "single":
        return f"{spec['frame_sample']} {spec['num_frames']} frames"
    return (
        f"sliding windows of {NATIVE_FRAMES.get(model, 8)}, stride {spec['stride']}, "
        f"{spec['sample_fps']:g} fps, cap {spec['max_frames']} frames"
    )


def result_path(dataset, model, subset, protocol, pool):
    return RESULTS_DIR / dataset / f"{model}_{subset}_{protocol}_{pool}.json"


def expected_paths(dataset, model, subset, spec):
    pools = ["none"] if spec["type"] == "single" else list(POOLS)
    return [result_path(dataset, model, subset, spec["name"], pool) for pool in pools]


def is_complete(path):
    try:
        return json.loads(path.read_text(encoding="utf-8")).get("status") == "complete"
    except Exception:
        return False


def save_row(dataset, model, subset, spec, pool, metrics=None, error=None, cached=False):
    row = {
        "group": ENV_GROUP,
        "dataset": dataset,
        "subset": subset,
        "model": model,
        "protocol": spec["name"],
        "protocol_type": spec["type"],
        "pool": pool,
        "sampling": sampling_text(model, spec),
        "gallery_cached": bool(cached),
        "status": "failed" if error else "complete",
        "created_at": datetime.now(timezone.utc).isoformat(),
    }
    if metrics is not None:
        row.update({key: float(metrics[key]) for key in METRIC_KEYS if key in metrics})
    if error is not None:
        row["error"] = str(error)[:2000]
    path = result_path(dataset, model, subset, spec["name"], pool)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(row, indent=2), encoding="utf-8")


def clean_cuda():
    gc.collect()
    try:
        import torch

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:
        pass


def run_one(encoder, model, dataset, subset, spec):
    paths = expected_paths(dataset, model, subset, spec)
    if all(is_complete(path) for path in paths):
        print(f"skip {model} {dataset}/{subset} {spec['name']}", flush=True)
        return
    splits = load_split(dataset, subset)
    cache = WORK / "frame_cache" / dataset
    batch = MODEL_BATCH.get(model, BATCH)
    text_batch = MODEL_TEXT_BATCH.get(model, TEXT_BATCH)
    print(f"== {PRETTY.get(model, model)} | {dataset}/{subset} | {spec['name']} ({sampling_text(model, spec)})", flush=True)
    try:
        if spec["type"] == "single":
            scored = {
                "none": evaluate_text_to_tracklet(
                    encoder,
                    splits,
                    num_frames=spec["num_frames"],
                    batch_size=batch,
                    text_batch_size=text_batch,
                    junk_same_camera=False,
                    frame_cache=cache,
                    frame_sample=spec["frame_sample"],
                )
            }
        else:
            scored = evaluate_text_to_tracklet_windows(
                encoder,
                splits,
                num_frames=NATIVE_FRAMES.get(model, 8),
                stride=spec["stride"],
                sample_fps=spec["sample_fps"],
                max_frames=spec["max_frames"],
                pools=POOLS,
                batch_size=batch,
                text_batch_size=text_batch,
                junk_same_camera=False,
                frame_cache=cache,
            )
    except RuntimeError as exc:
        pools = ["none"] if spec["type"] == "single" else list(POOLS)
        for pool in pools:
            save_row(dataset, model, subset, spec, pool, error=exc)
        if "out of memory" in str(exc).lower():
            raise StopModel(str(exc)) from exc
        raise
    for pool, metrics in scored.items():
        save_row(dataset, model, subset, spec, pool, metrics=metrics, cached=encoder.last_hit)
        print(f"-- {pool}: R1 {metrics['Rank-1']:.2f}  mAP {metrics['mAP']:.2f}  mINP {metrics.get('mINP', float('nan')):.2f}", flush=True)

In [ ]:
ensure_cuda_healthy(DEVICE)

for model in MODELS:
    jobs = [
        (dataset, spec, subset)
        for dataset in RUN_DATASETS
        for spec in protocols_for(model, dataset)
        for subset in DATASETS[dataset]["subsets"]
    ]
    todo = [job for job in jobs if not all(is_complete(p) for p in expected_paths(job[0], model, job[2], job[1]))]
    print(f"\n######## {PRETTY.get(model, model)}: {len(todo)} of {len(jobs)} runs to do", flush=True)
    if not todo:
        continue
    try:
        encoder = GalleryCache(build_encoder(model, device=DEVICE))
    except Exception:
        traceback.print_exc()
        print(f"Could not build {model}; skipping it", flush=True)
        clean_cuda()
        continue
    try:
        for dataset, spec, subset in todo:
            run_one(encoder, model, dataset, subset, spec)
    except StopModel as exc:
        print(f"CUDA OOM on {model}; stopping this model: {exc}", flush=True)
    finally:
        del encoder
        clean_cuda()

print("Done", flush=True)

## 5. Tables

Reads every results folder it can find: this session's `zero_shot_results/`, plus any attached Kaggle inputs that contain a `zero_shot_results` folder (other groups, earlier runs). Each table is shown here and saved as CSV and Markdown under `zero_shot_results/tables/`.

Accuracy cells use each model's **best Rank-1 setting** per dataset subset (protocol and pool are shown in the settings table). All numbers are percentages except MdR (median rank).

In [ ]:
import numpy as np

KEY = ["dataset", "subset", "model", "protocol", "pool"]
LEGACY_TVPREID = {"prid", "ilids", "duke"}


def collect_rows():
    roots = [RESULTS_ROOT]
    if Path("/kaggle/input").is_dir():
        roots += [p for p in Path("/kaggle/input").rglob("zero_shot_results") if p.is_dir()]
    rows = []
    for root in dict.fromkeys(roots):
        for path in root.rglob("*.json"):
            try:
                row = json.loads(path.read_text(encoding="utf-8"))
            except Exception:
                continue
            if not isinstance(row, dict) or "Rank-1" not in row:
                continue
            if "dataset" not in row and row.get("subset") in LEGACY_TVPREID:
                row["dataset"] = "tvpreid"
                row.setdefault("protocol_type", "single" if row.get("pool") == "none" else "window")
            if row.get("status", "complete") != "complete" or "dataset" not in row:
                continue
            rows.append(row)
    if not rows:
        return pd.DataFrame(
            columns=KEY + ["Rank-1", "mAP", "mINP", "protocol_type", "gallery_cached", "Model", "Dataset",
                           "video_ms_per_item", "peak_gpu_gb", "total_s"]
        )
    frame = pd.DataFrame(rows)
    if "protocol_type" not in frame:
        frame["protocol_type"] = np.where(frame["pool"] == "none", "single", "window")
    frame["protocol_type"] = frame["protocol_type"].fillna(
        pd.Series(np.where(frame["pool"] == "none", "single", "window"), index=frame.index)
    )
    frame["created_at"] = frame.get("created_at", pd.Series(dtype=str)).fillna("")
    frame["gallery_cached"] = frame.get("gallery_cached", False)
    frame["gallery_cached"] = frame["gallery_cached"].fillna(False).astype(bool)
    frame = frame.sort_values("created_at").drop_duplicates(KEY, keep="last")
    frame["Model"] = frame["model"].map(lambda m: PRETTY.get(m, m))
    frame["Dataset"] = frame["dataset"].map(lambda d: DATASETS.get(d, {}).get("label", d))
    return frame.reset_index(drop=True)


RESULTS = collect_rows()
print(f"{len(RESULTS)} result rows from {RESULTS['model'].nunique() if len(RESULTS) else 0} models", flush=True)
BEST = (
    RESULTS.sort_values(["Rank-1", "mAP"], ascending=False, kind="mergesort")
    .groupby(["dataset", "subset", "model"], as_index=False)
    .head(1)
    .reset_index(drop=True)
)
TABLES = {}


def column_order(dataset):
    return [s for s in DATASETS.get(dataset, {}).get("subsets", []) if s in set(BEST.loc[BEST.dataset == dataset, "subset"])]


def setting_text(row):
    return row["protocol"] if row["pool"] in ("none", None) or pd.isna(row["pool"]) else f"{row['protocol']} / {row['pool']}"

### 5.1 Leaderboard per dataset (Rank-1 / mAP / mINP)

In [ ]:
for dataset in [d for d in DATASETS if d in set(BEST["dataset"])]:
    part = BEST[BEST.dataset == dataset].copy()
    subsets = column_order(dataset)
    part["cell"] = part.apply(lambda r: f"{r['Rank-1']:.2f} / {r['mAP']:.2f} / {r.get('mINP', np.nan):.2f}", axis=1)
    part["setting"] = part.apply(setting_text, axis=1)
    order = part.groupby("Model")["Rank-1"].mean().sort_values(ascending=False).index
    board = part.pivot(index="Model", columns="subset", values="cell").reindex(index=order, columns=subsets)
    board["mean Rank-1"] = part.groupby("Model")["Rank-1"].mean().reindex(order).round(2)
    settings = part.pivot(index="Model", columns="subset", values="setting").reindex(index=order, columns=subsets)
    label = DATASETS[dataset]["label"]
    TABLES[f"leaderboard_{dataset}"] = board
    TABLES[f"best_setting_{dataset}"] = settings
    display(Markdown(f"#### {label}"))
    display(board)
    display(Markdown(f"Best setting per cell ({label})"))
    display(settings)

### 5.2 Cross-dataset Rank-1 matrix

In [ ]:
if len(BEST):
    BEST["column"] = BEST.apply(lambda r: f"{DATASETS.get(r['dataset'], {}).get('label', r['dataset'])} {r['subset']}", axis=1)
    columns = [
        f"{DATASETS[d]['label']} {s}"
        for d in DATASETS
        for s in column_order(d)
    ]
    matrix = BEST.pivot(index="Model", columns="column", values="Rank-1").reindex(columns=columns)
    matrix["Mean"] = matrix.mean(axis=1)
    matrix = matrix.sort_values("Mean", ascending=False).round(2)
    TABLES["cross_dataset_rank1"] = matrix
    try:
        display(matrix.style.format("{:.2f}", na_rep="-").background_gradient(cmap="Greens", axis=0))
    except Exception:
        display(matrix)

### 5.3 Protocol and pool comparison (top three models per video dataset, Rank-1)

In [ ]:
for dataset in [d for d in DATASETS if DATASETS[d]["protocols"] == "video" and d in set(RESULTS["dataset"])]:
    part = RESULTS[RESULTS.dataset == dataset].copy()
    top = part.groupby("Model")["Rank-1"].max().sort_values(ascending=False).head(3).index
    part = part[part.Model.isin(top)]
    part["pool"] = part["pool"].replace({"none": "-"})
    table = part.pivot_table(index=["Model", "protocol", "pool"], columns="subset", values="Rank-1", aggfunc="max")
    table = table.reindex(columns=column_order(dataset)).round(2)
    table = table.reindex(top, level=0)
    TABLES[f"protocols_{dataset}"] = table
    display(Markdown(f"#### {DATASETS[dataset]['label']}"))
    display(table)

### 5.4 GroOT-MOT17: appearance vs action vs combined captions

Same gallery, different queries. A large positive gap means the model matches clothing and looks, not motion. Action captions repeat a lot across tracks ("a person walking on the street"), so low action Rank-1 is expected for every model.

In [ ]:
groot = BEST[BEST.dataset == "groot_mot17"]
if len(groot):
    gap = groot.pivot(index="Model", columns="subset", values="Rank-1").reindex(columns=["appearance", "action", "combined", "all"])
    gap["appearance - action"] = gap["appearance"] - gap["action"]
    gap["combined - appearance"] = gap["combined"] - gap["appearance"]
    gap = gap.sort_values("appearance", ascending=False).round(2)
    TABLES["groot_appearance_vs_action"] = gap
    display(gap)
else:
    print("No GroOT-MOT17 rows yet")

### 5.5 Speed and GPU (Tesla T4)

In [ ]:
if len(RESULTS):
    fresh = RESULTS[~RESULTS["gallery_cached"]]
    single = fresh[fresh["protocol_type"] == "single"]
    windows = fresh[fresh["protocol_type"] == "window"]
    speed = pd.DataFrame(
        {
            "ms / clip (single clip)": single.groupby("Model")["video_ms_per_item"].median(),
            "ms / clip (windows)": windows.groupby("Model")["video_ms_per_item"].median(),
            "peak GPU GB": RESULTS.groupby("Model")["peak_gpu_gb"].max(),
            "total minutes": RESULTS.groupby("Model")["total_s"].sum() / 60.0,
            "rows": RESULTS.groupby("Model").size(),
        }
    )
    speed = speed.sort_values("ms / clip (single clip)").round(2)
    TABLES["speed_gpu"] = speed
    display(speed)

### 5.6 Save tables

In [ ]:
def markdown_table(frame):
    frame = frame.reset_index() if not isinstance(frame.index, pd.RangeIndex) else frame
    headers = [str(c) for c in frame.columns]

    def cell(value):
        if isinstance(value, float):
            return "-" if np.isnan(value) else f"{value:.2f}"
        return str(value)

    lines = ["| " + " | ".join(headers) + " |", "|" + "|".join("---" for _ in headers) + "|"]
    for _, row in frame.iterrows():
        lines.append("| " + " | ".join(cell(v) for v in row) + " |")
    return "\n".join(lines)


TABLE_DIR = RESULTS_ROOT / "tables"
TABLE_DIR.mkdir(parents=True, exist_ok=True)
RESULTS.to_csv(TABLE_DIR / "all_rows.csv", index=False)
DATASET_SUMMARY.to_csv(TABLE_DIR / "datasets.csv", index=False)
report = ["# Zero-shot results", "", "## Datasets", "", markdown_table(DATASET_SUMMARY.drop(columns=["root"]))]
for name, table in TABLES.items():
    table.to_csv(TABLE_DIR / f"{name}.csv")
    report += ["", f"## {name}", "", markdown_table(table)]
(TABLE_DIR / "report.md").write_text("\n".join(report) + "\n", encoding="utf-8")
print("Wrote", sorted(p.name for p in TABLE_DIR.iterdir()), flush=True)

## Reference: earlier TVPReid run (base group)

Tesla T4, TVPReid test. Each accuracy cell is **Rank-1 / mAP / MdR** for that model's best Rank-1 on the subset. Milliseconds per clip and peak GPU are stable across protocols. Duke video encode is the wall clock of the run that set that model's best Duke Rank-1.

### Rank-1 / mAP / median rank

| Model | PRID | iLIDS | Duke |
|---|---:|---:|---:|
| IRRA | **66.55** / 76.09 / 1 | **30.00** / 41.27 / 5 | **38.06** / 50.79 / 2 |
| Perception Encoder L/14 | 38.38 / 52.04 / 2 | 19.33 / 32.55 / 6 | 17.66 / 28.12 / 11 |
| SigLIP 2 So400m | 35.56 / 50.84 / 3 | 18.67 / 28.88 / 10 | 21.97 / 33.03 / 7 |
| InternVideo2 CLIP-S | 32.39 / 43.77 / 4 | 13.33 / 24.14 / 11.5 | 18.49 / 28.36 / 12 |
| LanguageBind | 16.20 / 24.82 / 16 | 12.67 / 22.09 / 14 | 7.30 / 14.46 / 40.5 |
| X-CLIP | 4.93 / 10.05 / 43.5 | 5.33 / 10.52 / 38.5 | 1.41 / 3.95 / 164 |
| InternVideo2-1B-s2 | 4.23 / 10.78 / 34.5 | 4.00 / 10.26 / 29 | 1.82 / 3.82 / 159 |

### Setting, speed, and GPU

| Model | Best setting | ms / clip | Peak GPU | Duke video encode |
|---|---|---:|---:|---:|
| IRRA | PRID 8 fps `mean_s8`; iLIDS 2 fps `mean`; Duke 8 fps `query_max` | 34 | 0.31 GB | 4.8 min |
| Perception Encoder L/14 | PRID 8 fps `mean_s8`; iLIDS and Duke 8 fps `query_max` | 238 | 1.32 GB | 33.8 min |
| SigLIP 2 So400m | 8 fps `query_max` on all three subsets | 421 | 2.23 GB | 59.8 min |
| InternVideo2 CLIP-S | PRID 2 fps `max`; iLIDS and Duke 8 fps `query_max` | 354 | 1.54 GB | 50.2 min |
| LanguageBind | PRID 2 fps `mean`; iLIDS and Duke 8 fps `mean` | 192 | 1.22 GB | 27.2 min |
| X-CLIP | PRID and Duke 8 fps `mean`; iLIDS 2 fps `mean` | 30 | 0.45 GB | 4.2 min |
| InternVideo2-1B-s2 | PRID uniform 4; iLIDS 2 fps `query_max`; Duke 1 fps `mean_s8` | 262 | 2.81 GB | 6.9 min |